In [0]:
# ============================================================
# NOTEBOOK 00 — ANONIMIZADOR
# Vermont School - Pipeline V2
# 
# FUNCIÓN: Reemplaza nombres reales por códigos anónimos
# INPUT:   Archivos XLS originales en bronze/raw/
# OUTPUT:  Archivos XLS anonimizados en bronze/anon/
#          tabla_mapeo.csv en /privado/ (NUNCA sale de Databricks)
#
# ⚠️  MODO_ANONIMO = False → uso real en Vermont (nombres reales)
# ============================================================

MODO_ANONIMO = True  # Cambiar a False para uso interno Vermont

BRONZE  = "/Volumes/workspace/vermont/bronze"
PRIVADO = "/Volumes/workspace/vermont/privado"  # NO va a GitHub

# Crear carpetas si no existen
import os
for carpeta in [
    f"{BRONZE}/raw/24_25",
    f"{BRONZE}/raw/25_26", 
    f"{BRONZE}/anon/24_25",
    f"{BRONZE}/anon/25_26",
    f"{PRIVADO}",
]:
    dbutils.fs.mkdirs(carpeta)
    
print("✓ Estructura de carpetas creada")
print(f"\n  {BRONZE}/raw/24_25/    ← subir archivos XLS 24-25 aquí")
print(f"  {BRONZE}/raw/25_26/    ← subir archivos XLS 25-26 aquí")
print(f"  {BRONZE}/anon/24_25/   ← archivos anonimizados")
print(f"  {BRONZE}/anon/25_26/   ← archivos anonimizados")
print(f"  {PRIVADO}/             ← tabla de mapeo (privado)")

In [0]:
%pip install lxml

In [0]:
# CELDA 2 — Construir tabla de mapeo completa
import pandas as pd
from io import StringIO
import hashlib, random, re

random.seed(42)

RAW_24 = f"{BRONZE}/raw/24_25"
RAW_25 = f"{BRONZE}/raw/25_26"

def leer_html(ruta):
    with open(ruta, "r", encoding="utf-8", errors="replace") as f:
        return pd.read_html(StringIO(f.read()))

def leer_csv(ruta):
    return pd.read_csv(ruta, sep=";", encoding="utf-8-sig")

def norm_nombre(x):
    return str(x).replace(',', '').strip()

# Recolectar todos los estudiantes de ambos años
print("Recolectando estudiantes...")

# Desde asistencia (tiene código + nombre + sección)
da24 = leer_html(f"{RAW_24}/24_25_asistencia.xls")[0]
da25 = leer_html(f"{RAW_25}/25_26_asistencia.xls")[0]

da24 = da24[da24['Estudiante'] != 'Total'].copy()
da25 = da25[da25['Estudiante'] != 'Total'].copy()

# Construir tabla maestra de estudiantes
estudiantes = {}

for da, año in [(da24, '24_25'), (da25, '25_26')]:
    for _, row in da.iterrows():
        cod = str(row['Código']).strip()
        nombre = norm_nombre(row['Estudiante'])
        seccion = str(row['Matrícula']).strip()
        if cod not in estudiantes:
            estudiantes[cod] = {
                'codigo_real': cod,
                'nombre_real': nombre,
            }
        estudiantes[cod][f'seccion_{año}'] = seccion

# Recolectar docentes de convivencia ambos años
print("Recolectando docentes...")
dfs_conv = []
for ruta in [
    f"{RAW_24}/24_25_convivencia_t1.xls",
    f"{RAW_24}/24_25_convivencia_t2.xls",
    f"{RAW_25}/25_26_convivencia_tipo1.xls",
    f"{RAW_25}/25_26_convivencia_tipo2.xls",
]:
    dfs_conv.append(leer_csv(ruta))
df_conv_all = pd.concat(dfs_conv, ignore_index=True)

# Docentes conocidos de Middle School
DOCENTES_MIDDLE = [
    "ACUÑA BARRAGAN JULIO CESAR",
    "RODRIGUEZ RODRIGUEZ ANDRES IGNACIO",
    "Huang Huang Wen Zhun",
    "Villa Uribe Veronica",
    "Duarte Alba Ingrid Paola",
    "Marsiglia Lopez Dalma Elizabeth",
    "Posada Penagos Alejandro",
    "POSADA RESTREPO DEISI YULIANA",
    "Gomez Mejia Didier Johan",
    "Velasco Hernandez Andres Felipe",
    "Villegas Cardona Fransua",
    "Lopez Vargas Francisco Antonio",
    "MOLINA PACHON LORENA ALEXANDRA",
    "TABOADA DURANGO JUAN ANTONIO",
    "Agudelo Rodriguez Elmer Ivar",
    "Arboleda Patiño Cipriano",
    "TORRES POSADA CRISTIAN CAMILO",
    "Zapata Mejía Luis Camilo",
    "Angela Gomez",
    "Carlos Garcia",
]
OTROS_ROLES = {
    "AGUDELO SERNA /RETIRADO SEBASTIÁN": "retirado",
    "Mercado Diaz Darwin Raul":          "retirado",
    "Brito Londoño Edwin":               "coordinador",
    "SELIGMAN DIEGO":                    "psicologo_hs",
    "Sanchez Rios Valentina":            "otro",
}

# Todos los autores en convivencia
todos_autores = set(df_conv_all['Autor'].dropna().unique())
codigos_doc = [f'P{str(i).zfill(2)}' for i in range(1, len(DOCENTES_MIDDLE)+1)]
random.shuffle(codigos_doc)
docente_map = {nombre: cod for nombre, cod in zip(DOCENTES_MIDDLE, codigos_doc)}
for nombre, rol in OTROS_ROLES.items():
    docente_map[nombre] = rol
# Cualquier autor no identificado
for autor in todos_autores:
    if autor not in docente_map:
        docente_map[autor] = 'otro'

# Secciones — recodificación aleatoria
seccion_map = {}
for g in ['7', '8', '9']:
    l = ['SA', 'SB']; random.shuffle(l)
    seccion_map[f'{g}° A'] = f'G{g}{l[0]}'
    seccion_map[f'{g}° B'] = f'G{g}{l[1]}'

print(f"\n✓ Estudiantes únicos encontrados: {len(estudiantes)}")
print(f"✓ Docentes mapeados: {len(docente_map)}")
print(f"✓ Secciones: {seccion_map}")

In [0]:
# CELDA 3 — Guardar tabla de mapeo privada
import json

# Construir tabla de mapeo completa
mapeo_estudiantes = []
for cod, info in estudiantes.items():
    mapeo_estudiantes.append({
        'codigo_real':    cod,
        'nombre_real':    info['nombre_real'],
        'seccion_24_25':  info.get('seccion_24_25', None),
        'seccion_25_26':  info.get('seccion_25_26', None),
        'seccion_anon_24_25': seccion_map.get(info.get('seccion_24_25',''), None),
        'seccion_anon_25_26': seccion_map.get(info.get('seccion_25_26',''), None),
    })

df_mapeo_est = pd.DataFrame(mapeo_estudiantes)
df_mapeo_doc = pd.DataFrame([
    {'nombre_real': k, 'codigo_anon': v} 
    for k, v in docente_map.items()
])
df_mapeo_sec = pd.DataFrame([
    {'seccion_real': k, 'seccion_anon': v}
    for k, v in seccion_map.items()
])

# Guardar en privado
df_mapeo_est.to_csv(f"{PRIVADO}/mapeo_estudiantes.csv", index=False)
df_mapeo_doc.to_csv(f"{PRIVADO}/mapeo_docentes.csv", index=False)
df_mapeo_sec.to_csv(f"{PRIVADO}/mapeo_secciones.csv", index=False)

# Guardar también los mapeos como variables para usar en celdas siguientes
with open(f"{PRIVADO}/seccion_map.json", 'w') as f:
    json.dump(seccion_map, f)
with open(f"{PRIVADO}/docente_map.json", 'w') as f:
    json.dump(docente_map, f)

print("=" * 55)
print("TABLA DE MAPEO GUARDADA EN PRIVADO")
print("=" * 55)
print(f"✓ mapeo_estudiantes.csv — {len(df_mapeo_est)} estudiantes")
print(f"✓ mapeo_docentes.csv    — {len(df_mapeo_doc)} docentes")
print(f"✓ mapeo_secciones.csv  — {len(df_mapeo_sec)} secciones")
print(f"\n⚠️  Estos archivos NUNCA salen de Databricks")
print(f"   Ruta: {PRIVADO}/")
print(f"\nPreview mapeo estudiantes:")
print(df_mapeo_est.head(5).to_string(index=False))

In [0]:
# CELDA 4 — Función anonimizadora y aplicación a archivos

def anonimizar_nombre_estudiante(nombre_real, df_mapeo):
    """Reemplaza nombre real por código de matrícula"""
    # Buscar por nombre normalizado
    nombre_norm = norm_nombre(nombre_real)
    match = df_mapeo[df_mapeo['nombre_real'] == nombre_norm]
    if len(match) > 0:
        return match.iloc[0]['codigo_real']
    return nombre_real  # si no encuentra, deja igual

def anonimizar_df_notas(tables, año, df_mapeo):
    """Anonimiza las tablas de notas"""
    cursos = ["7° A","7° B","8° A","8° B","9° A","9° B"]
    tables_anon = []
    for tabla, curso in zip(tables, cursos):
        tabla = tabla.copy()
        tabla.columns = [c[1] if isinstance(c,tuple) else c 
                        for c in tabla.columns]
        cn = tabla.columns[0]
        # Reemplazar nombres por códigos
        if MODO_ANONIMO:
            tabla[cn] = tabla[cn].apply(
                lambda x: anonimizar_nombre_estudiante(x, df_mapeo) 
                if pd.notna(x) else x
            )
        tables_anon.append((tabla, curso))
    return tables_anon

def anonimizar_df_asistencia(da, año, df_mapeo):
    """Anonimiza el dataframe de asistencia"""
    da = da.copy()
    da = da[da['Estudiante'] != 'Total']
    if MODO_ANONIMO:
        da['Estudiante'] = da['Estudiante'].apply(
            lambda x: anonimizar_nombre_estudiante(x, df_mapeo)
        )
        da['Matrícula'] = da['Matrícula'].map(seccion_map).fillna(da['Matrícula'])
    return da

def anonimizar_df_conv(dc, df_mapeo):
    """Anonimiza el dataframe de convivencia"""
    dc = dc.copy()
    if MODO_ANONIMO:
        dc['Persona'] = dc['Persona'].apply(
            lambda x: anonimizar_nombre_estudiante(x, df_mapeo)
            if pd.notna(x) else x
        )
        dc['Autor'] = dc['Autor'].map(docente_map).fillna('otro')
        dc['último Editor'] = dc['último Editor'].map(docente_map).fillna('otro')
        if 'Sección' in dc.columns:
            dc['Sección'] = dc['Sección'].map(seccion_map).fillna(dc['Sección'])
        # Eliminar columnas con texto libre (descripciones identificables)
        cols_eliminar = [c for c in dc.columns if c in [
            'Descripcion especifica de la situacion o del acompanamiento:',
            'Estrategias pedagogicas implementadas:',
            'Otras estrategias que se haya Implementado para atender la situacion:',
            'Comentarios adicionales',
            'Documento Soporte'
        ]]
        dc = dc.drop(columns=cols_eliminar, errors='ignore')
    return dc

# Aplicar anonimización a 24-25
print("Anonimizando 24-25...")
tables_24 = leer_html(f"{RAW_24}/24_25_notas.xls")
da24_raw  = leer_html(f"{RAW_24}/24_25_asistencia.xls")[0]
ct1_raw   = leer_csv(f"{RAW_24}/24_25_convivencia_t1.xls")
ct2_raw   = leer_csv(f"{RAW_24}/24_25_convivencia_t2.xls")

tables_24_anon = anonimizar_df_notas(tables_24, '24_25', df_mapeo_est)
da24_anon      = anonimizar_df_asistencia(da24_raw, '24_25', df_mapeo_est)
ct1_anon       = anonimizar_df_conv(ct1_raw, df_mapeo_est)
ct2_anon       = anonimizar_df_conv(ct2_raw, df_mapeo_est)

# Aplicar anonimización a 25-26
print("Anonimizando 25-26...")
tables_25  = leer_html(f"{RAW_25}/25_26_notas.xls")
da25_raw   = leer_html(f"{RAW_25}/25_26_asistencia.xls")[0]
c25t1_raw  = leer_csv(f"{RAW_25}/25_26_convivencia_tipo1.xls")
c25t2_raw  = leer_csv(f"{RAW_25}/25_26_convivencia_tipo2.xls")

tables_25_anon = anonimizar_df_notas(tables_25, '25_26', df_mapeo_est)
da25_anon      = anonimizar_df_asistencia(da25_raw, '25_26', df_mapeo_est)
c25t1_anon     = anonimizar_df_conv(c25t1_raw, df_mapeo_est)
c25t2_anon     = anonimizar_df_conv(c25t2_raw, df_mapeo_est)

print("\n✓ Anonimización completa")
print(f"\nVerificación 24-25 notas (primeros 3 nombres):")
print(tables_24_anon[0][0][tables_24_anon[0][0].columns[0]].dropna().unique()[:3])
print(f"\nVerificación 25-26 asistencia (primeras 3 filas):")
print(da25_anon[['Estudiante','Matrícula']].head(3).to_string(index=False))

In [0]:
# CELDA 5 — Guardar archivos anonimizados en Bronze/anon/

ANON_24 = f"{BRONZE}/anon/24_25"
ANON_25 = f"{BRONZE}/anon/25_26"

# ── Guardar notas como CSV (una por sección) ──
cursos = ["7A","7B","8A","8B","9A","9B"]

for (tabla, curso), nombre_curso in zip(tables_24_anon, cursos):
    tabla.to_csv(f"{ANON_24}/notas_{nombre_curso}.csv", index=False)

for (tabla, curso), nombre_curso in zip(tables_25_anon, cursos):
    tabla.to_csv(f"{ANON_25}/notas_{nombre_curso}.csv", index=False)

print("✓ Notas guardadas")

# ── Guardar asistencia ──
# Columnas comunes entre años (ignorar columnas extra de 24-25)
cols_asist = ['Estudiante', 'Código', 'Matrícula',
              'Ausencia Clase', 'Ausencia Clase con Excusa',
              'Ausencia Colegio', 'Ausencia Colegio con Excusa',
              'Retardo', 'Salida Temprano', 'Total']

da24_anon[[c for c in cols_asist if c in da24_anon.columns]]\
    .to_csv(f"{ANON_24}/asistencia.csv", index=False)
da25_anon[[c for c in cols_asist if c in da25_anon.columns]]\
    .to_csv(f"{ANON_25}/asistencia.csv", index=False)

print("✓ Asistencia guardada")

# ── Guardar convivencia ──
# Combinar T1 y T2 de 24-25 en uno solo
conv_24 = pd.concat([ct1_anon, ct2_anon], ignore_index=True)
conv_24.to_csv(f"{ANON_24}/convivencia.csv", index=False)

# 25-26 separados por tipo
c25t1_anon.to_csv(f"{ANON_25}/convivencia_tipo1.csv", index=False)
c25t2_anon.to_csv(f"{ANON_25}/convivencia_tipo2.csv", index=False)

print("✓ Convivencia guardada")

# ── Verificar archivos guardados ──
print("\n=== ARCHIVOS EN BRONZE/ANON ===")
for año in ['24_25', '25_26']:
    ruta = f"{BRONZE}/anon/{año}"
    archivos = os.listdir(ruta)
    print(f"\n  {año}: {len(archivos)} archivos")
    for a in sorted(archivos):
        size = os.path.getsize(f"{ruta}/{a}")
        print(f"    {a:35} {size:>10,} bytes")

print(f"\n✓ Anonimizador completo")
print(f"  Modo anónimo: {MODO_ANONIMO}")
print(f"  Mapeo privado en: {PRIVADO}/")
print(f"  Datos anónimos en: {BRONZE}/anon/")